<a href="https://colab.research.google.com/github/soleildayana/Data-Mining/blob/main/StellarMovingGroups_AgeEstimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install astroquery -Uq

## Metodología

### Selección de los 3 Grupos Móviles (Con edad <20 Myr, excepto LLC)

Para este estudio, lo más conveniente es escoger 3 asociaciones cercanas y bien pobladas en Simbad/Gaia, porque para calcular $(x,y,z,v_x,v_y,v_z)$ se requiere la velocidad radial, cuyo valor sólo se reporta para cierta fracción de estrellas de Gaia DR3. Elegir un grupo pequeño o lejano puede implicar que, luego de filtrar queden muy pocas estrellas.
Por esto, luego de estudiar la edad típica y la distancia media en la lista de Wikipedia [List of nearby stellar associations and moving groups](https://en.wikipedia.org/wiki/List_of_nearby_stellar_associations_and_moving_groups), los candidatos seleccionados fueron:


1.   Elemento de lista
2.   Elemento de lista
3.   el

### Query (Simbad + Gaia TAP)



In [5]:
from astroquery.simbad import Simbad
from astroquery.gaia import Gaia

In [12]:
from astroquery.simbad import Simbad
my_simbad = Simbad()
my_simbad.server = "simbad.harvard.edu"
result_table = my_simbad.query_object("HD 8745")
print(result_table)

 main_id        ra            dec       ...     coo_bibcode     matched_id
               deg            deg       ...                               
--------- -------------- -------------- ... ------------------- ----------
HD   8745 21.72418762496 37.49220928213 ... 2020yCat.1350....0G  HD   8745


In [14]:
from astroquery.simbad import Simbad

my_simbad = Simbad()
my_simbad.server = "simbad.harvard.edu"

query_simbad_twa = """
SELECT DISTINCT
    basic.oid,
    basic.main_id,
    basic.ra,
    basic.dec,
    ident.id AS gaia_id
FROM basic
JOIN ident ON basic.oid = ident.oidref
WHERE basic.oid IN (
    SELECT oidref FROM ident WHERE id LIKE 'TWA %'
)
AND ident.id LIKE 'Gaia DR3 %'
"""

tabla_simbad_twa = my_simbad.query_tap(query_simbad_twa)
print(f"Miembros TWA en Simbad con Gaia DR3: {len(tabla_simbad_twa)}")
tabla_simbad_twa.write("simbad_twa.csv", format="csv", overwrite=True)

Miembros TWA en Simbad con Gaia DR3: 58


In [8]:
# cono alrededor del centro de TWA (RA ~ 165 deg, Dec ~ -34 deg, radio 25 deg)
# y paralaje entre 10 y 30 mas (distancia ~ 33 a 100 pc)
query_gaia = """
SELECT source_id, ra, dec, parallax, parallax_error,
       pmra, pmra_error, pmdec, pmdec_error,
       radial_velocity, radial_velocity_error, ruwe
FROM gaiadr3.gaia_source
WHERE 1 = CONTAINS(
    POINT('ICRS', ra, dec),
    CIRCLE('ICRS', 165.0, -34.0, 25.0)
)
AND parallax BETWEEN 10.0 AND 30.0
AND radial_velocity IS NOT NULL
"""

job = Gaia.launch_job_async(query_gaia)
tabla_gaia = job.get_results()
print(f"Estrellas en Gaia con velocidad radial en la región: {len(tabla_gaia)}")

INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]
Estrellas en Gaia con velocidad radial en la región: 5534


In [9]:
#tabla_gaia.write("gaia_twa.csv", format="csv", overwrite=True)

In [16]:
import pandas as pd

df_simbad = pd.read_csv("simbad_twa.csv")
df_gaia = pd.read_csv("gaia_twa.csv")

# Extraemos el número limpio de source_id quitando el texto 'Gaia DR3 '
ids_simbad = df_simbad["gaia_id"].str.replace("Gaia DR3 ", "", regex=False).astype("int64")

# Creamos la columna booleana de pertenencia confirmada según Simbad
df_gaia["es_miembro_simbad"] = df_gaia["source_id"].isin(ids_simbad)

print("Total estrellas en cono Gaia (con RV):", len(df_gaia))
print("Miembros confirmados de Simbad cruzados en Gaia:", df_gaia["es_miembro_simbad"].sum())

Total estrellas en cono Gaia (con RV): 5534
Miembros confirmados de Simbad cruzados en Gaia: 25


In [19]:
# Verificar cuántos de los IDs de Simbad NO cayeron dentro de los 5,534 del cono
faltantes = set(ids_simbad) - set(df_gaia["source_id"])
print(f"IDs de Simbad fuera del cono/paralaje: {len(faltantes)}")


IDs de Simbad fuera del cono/paralaje: 33
